In [1]:
import re
import numpy as np
import pandas as pd
import os

## Base Elondou

Configuración inicial y datos

In [2]:
path = r"C:\EDU\repositorios\AI-y-mercados-laborales-Ecuador" # Cambiar para replicar

onet = pd.read_csv(os.path.join(path,"data/full_labelset.tsv"), sep='\t', index_col=0)

Funciones para limpiar códigos

In [3]:
def norm_soc(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    m = re.search(r'(\d{2})[-\.]?(\d{4})', x)
    if m:
        return f"{m.group(1)}-{m.group(2)}"
    return np.nan

def norm_ciuo(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace(',', '.')
    m = re.search(r'(\d{4})', x)
    if m:
        return m.group(1)
    return np.nan

Creamos scores basados en humanos de forma explícita

In [4]:
def human_to_scores(label):
    if label == 'E1':
        return pd.Series([1.0, 1.0, 1.0])
    elif label == 'E2':
        return pd.Series([0.0, 0.5, 1.0])
    else:
        return pd.Series([0.0, 0.0, 0.0])

onet[['alpha_h', 'beta_h', 'gamma_h']] = onet['human_exposure_agg'].apply(human_to_scores)

Agregamos SOC

In [5]:
onet['soc_code'] = onet['O*NET-SOC Code'].apply(norm_soc)
onet['task_weight'] = np.where(onet['Task Type'].eq('Core'), 2.0, 1.0)

soc_exp = (
    onet.groupby(['soc_code', 'Title'], as_index=False)
        .apply(lambda g: pd.Series({
            'alpha_gpt': np.average(g['alpha'], weights=g['task_weight']),
            'beta_gpt': np.average(g['beta'], weights=g['task_weight']),
            'gamma_gpt': np.average(g['gamma'], weights=g['task_weight']),
            'alpha_h': np.average(g['alpha_h'], weights=g['task_weight']),
            'beta_h': np.average(g['beta_h'], weights=g['task_weight']),
            'gamma_h': np.average(g['gamma_h'], weights=g['task_weight']),
            'n_tasks': g['Task ID'].nunique()
        }))
        .reset_index(drop=True)
)

C:\Users\OSCARJ\AppData\Local\Temp\ipykernel_43728\783560945.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


## Crosswalk

In [6]:
cw_own = pd.read_csv(os.path.join(path, "crosswalk/crosswalkOwn.csv"), sep=',', encoding='latin1', dtype=str)
cw_bls = pd.read_csv(os.path.join(path, "crosswalk/crosswalkBLS.csv"), sep="\t", dtype=str)

In [7]:
cw_bls['soc_code'] = cw_bls['2010 SOC Code'].apply(norm_soc)
cw_bls['ciuo_code'] = cw_bls['ISCO-08 Code'].apply(norm_ciuo)   # only if this is truly equivalent in your Ecuador setup

Esposición CIUO

In [8]:
soc_ciuo = cw_bls.merge(soc_exp, on='soc_code', how='left')

In [9]:
ciuo_exp = (
    soc_ciuo.groupby('ciuo_code', as_index=False)
    .agg({
        'alpha_gpt': 'mean',
        'beta_gpt': 'mean',
        'gamma_gpt': 'mean',
        'alpha_h': 'mean',
        'beta_h': 'mean',
        'gamma_h': 'mean',
        'soc_code': 'nunique'
    })
    .rename(columns={'soc_code': 'n_soc_matches'})
)

## Merge ENEMDU

In [12]:
path_enemdu = r"Z:\survey\ECU\ENEMDU\2025\m12\data_orig\enemdu_persona_2025_12.dta" # Cambiar para replicar

enemdu = pd.read_stata(path_enemdu, convert_categoricals=False)

In [13]:
enemdu['ciuo_code'] = enemdu['p40'].apply(norm_ciuo)
enemdu_exp = enemdu.merge(ciuo_exp, on='ciuo_code', how='left')

C:\Users\OSCARJ\AppData\Local\Temp\ipykernel_43728\712123712.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  enemdu['ciuo_code'] = enemdu['p40'].apply(norm_ciuo)


In [20]:
enemdu_exp.to_stata(os.path.join(path, r'data\enemdu_con_tareas.parquet'))